# AlphaPulse — Week 1: Data Acquisition & Integrity Check

**Project:** Infotact Solutions Project 2 (Portfolio Risk & Volatility Monitor)  
**Primary Source:** Twelve Data API (with `yfinance` fallback)  
**Targets:** Local Parquet/CSV + **MongoDB Compass** (`alphapulse.market_prices`) + **Hugging Face Hub**

In [1]:
import sys
from pathlib import Path

# Ensure project root is on sys.path when running inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import time

import polars as pl

from scripts.week1_ingest import (
    align_and_clean,
    load_asset_series,
    push_to_huggingface,
    run_integrity_checks,
    save_to_mongodb,
)
from src.config import FX_ASSET, PORTFOLIO_ASSETS, PROCESSED_DIR, get_settings

settings = get_settings()
print(f"Loaded settings | Mongo DB: {settings.mongo_db} | HF Repo: {settings.hf_repo_id}")
td_ok = bool(settings.twelve_data_api_key and "your_" not in settings.twelve_data_api_key)
print(f"Twelve Data key configured: {td_ok}")

Loaded settings | Mongo DB: alphapulse | HF Repo: RS2226/alphapulse-market-data
Twelve Data key configured: True


## 1. Fetch Historical OHLCV Data (Twelve Data + Fallback)
Pulls 5 years (`1260` trading days) of corporate-action-adjusted daily bars for `USD/INR`, Indian NSE equities, Gold BeES, and `BTC/USD`.

In [3]:
fx_df = load_asset_series(FX_ASSET, settings)
asset_frames = []
for asset in PORTFOLIO_ASSETS:
    df_asset = load_asset_series(asset, settings)
    asset_frames.append(df_asset)
    time.sleep(1.0)

print(f"Fetched FX ({fx_df.height} rows) + {len(asset_frames)} portfolio assets.")

2026-11-27 19:34:19,667 | WARNING | Twelve Data request failed for USDINR: [WinError 10054] An existing connection was forcibly closed by the remote host
2026-11-27 19:34:19,671 | INFO | Falling back to yfinance for USDINR (USDINR=X)
2026-11-27 19:34:26,193 | WARNING | Twelve Data HTTP 404 for TCS (TCS); falling back to yfinance.
2026-11-27 19:34:26,194 | INFO | Falling back to yfinance for TCS (TCS.NS)
2026-11-27 19:34:30,237 | INFO | Fetched INFY via Twelve Data (1260 rows)
2026-11-27 19:34:31,711 | WARNING | Twelve Data HTTP 404 for HDFCBANK (HDFCBANK); falling back to yfinance.
2026-11-27 19:34:31,713 | INFO | Falling back to yfinance for HDFCBANK (HDFCBANK.NS)
2026-11-27 19:34:33,656 | WARNING | Twelve Data HTTP 404 for RELIANCE (RELIANCE); falling back to yfinance.
2026-11-27 19:34:33,657 | INFO | Falling back to yfinance for RELIANCE (RELIANCE.NS)
2026-11-27 19:34:35,709 | WARNING | Twelve Data HTTP 404 for SUNPHARMA (SUNPHARMA); falling back to yfinance.
2026-11-27 19:34:35,710

Fetched FX (1300 rows) + 7 portfolio assets.


## 2. INR Normalization, Forward-Fill Alignment & `float32` Optimization

In [4]:
cleaned_df = align_and_clean(asset_frames, fx_df)
run_integrity_checks(cleaned_df)

print(cleaned_df.schema)
print(f"Estimated memory: {cleaned_df.estimated_size('mb'):.3f} MB")
cleaned_df.group_by("asset_id", "sector", "source").agg(
    pl.len().alias("rows"),
    pl.col("date").min().alias("start_date"),
    pl.col("date").max().alias("end_date"),
    pl.col("close").min().alias("min_inr"),
    pl.col("close").max().alias("max_inr"),
).sort("asset_id")

2026-11-27 19:34:40,846 | INFO | Integrity check passed: 5950 total rows (850 days x 7 assets)


Schema([('date', Date), ('asset_id', String), ('sector', String), ('source', String), ('open', Float32), ('high', Float32), ('low', Float32), ('close', Float32), ('volume', Float32)])
Estimated memory: 0.274 MB


asset_id,sector,source,rows,start_date,end_date,min_inr,max_inr
str,str,str,u32,date,date,f32,f32
"""BTC_INR""","""Crypto""","""twelvedata""",850,2023-06-19,2026-11-27,2.0898e6,1.1061588e7
"""GOLDBEES""","""Commodities""","""yfinance""",850,2023-06-19,2026-11-27,48.25,146.529999
"""HDFCBANK""","""Financials""","""yfinance""",850,2023-06-19,2026-11-27,662.637817,996.4198
"""INFY""","""Technology""","""twelvedata""",850,2023-06-19,2026-11-27,963.718018,1861.057617
"""RELIANCE""","""Energy""","""yfinance""",850,2023-06-19,2026-11-27,1099.984863,1584.971802
"""SUNPHARMA""","""Healthcare""","""yfinance""",850,2023-06-19,2026-11-27,960.29425,2001.099976
"""TCS""","""Technology""","""yfinance""",850,2023-06-19,2026-11-27,1961.362061,4208.337891


## 3. Corporate Action & Split Continuity Audit

In [5]:
jumps = (
    cleaned_df.sort(["asset_id", "date"])
    .with_columns((pl.col("close") / pl.col("close").shift(1).over("asset_id") - 1.0).alias("daily_ret"))
    .filter(pl.col("daily_ret").abs() > 0.60)
)
print(f"Extreme single-day jumps (>60%): {jumps.height}")
if jumps.height > 0:
    display(jumps)

Extreme single-day jumps (>60%): 0


## 4. Persist Locally, Upsert to MongoDB & Push to Hugging Face

In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
parquet_path = PROCESSED_DIR / "market_prices.parquet"
csv_path = PROCESSED_DIR / "market_prices.csv"

cleaned_df.write_parquet(parquet_path)
cleaned_df.write_csv(csv_path)
print(f"Saved local Parquet & CSV to {PROCESSED_DIR}")

save_to_mongodb(cleaned_df, settings)
push_to_huggingface(str(parquet_path), settings)

Saved local Parquet & CSV to C:\Users\acer\Downloads\AlphaPulse\data\processed


2026-11-27 19:34:42,825 | INFO | Upserted 5950 records into MongoDB (alphapulse.market_prices)
2026-11-27 19:34:43,383 | INFO | HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 409 Conflict"
c:\Users\acer\Downloads\AlphaPulse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-11-27 19:34:43,794 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/RS2226/alphapulse-market-data/preupload/main "HTTP/1.1 200 OK"
Processing Files (1 / 1): 100%|██████████|  110kB /  110kB, 7.84kB/s  






Processing Files (1 / 1): 100%|██████████|  110kB /  110kB, 6.78kB/s  
New Data Upload: 100%|██████████|  110kB /  110kB, 6.78kB/s  
Validating: ██████████| 100%            
2026-11-27 19:35:22,094 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/RS2226/alphapulse-market-data/commi